# Build self-attention from scratch

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Implement scaled dot-product attention
- Add a causal mask
- Inspect real attention in a pretrained transformer

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)

## 1. Scaled dot-product attention

In [ ]:
def attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / Q.shape[-1] ** 0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w = F.softmax(scores, dim=-1)
    return w @ V, w
q = torch.tensor([[1., 0.]]); K = torch.tensor([[1., 0.], [0., 1.]]); V = torch.tensor([[10., 0.], [0., 10.]])
out, w = attention(q, K, V)
print(w, out)   # weights ≈ (0.67, 0.33)

## 2. Self-attention over a sequence, with a causal mask

In [ ]:
T, d = 5, 8
X = torch.randn(T, d)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
mask = torch.tril(torch.ones(T, T))
out, w = attention(X @ Wq, X @ Wk, X @ Wv, mask)
print(w.round(decimals=2))   # upper triangle is 0: no peeking at future tokens
print(torch.allclose(out, F.scaled_dot_product_attention((X@Wq)[None], (X@Wk)[None], (X@Wv)[None], is_causal=True)[0], atol=1e-5))

## 3. Real attention in a pretrained model

In [ ]:
!pip -q install transformers
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModel.from_pretrained('distilbert-base-uncased', output_attentions=True)
inputs = tok('The cat sat on the mat because it was tired', return_tensors='pt')
att = model(**inputs).attentions[-1][0].mean(0)   # last layer, averaged over heads
tokens = tok.convert_ids_to_tokens(inputs['input_ids'][0])
i = tokens.index('it')
print(sorted(zip(att[i].tolist(), tokens), reverse=True)[:5])

## Exercises
1. Write multi-head attention with 2 heads of size 4 by splitting `X @ Wq` into two parts.
2. Change the temperature of the softmax (divide scores by 0.5 or 2). How do the weights change?

In [ ]:
# your work here

## Solutions

In [ ]:
Qh = (X @ Wq).view(T, 2, 4).transpose(0, 1); Kh = (X @ Wk).view(T, 2, 4).transpose(0, 1); Vh = (X @ Wv).view(T, 2, 4).transpose(0, 1)
out_h, _ = attention(Qh, Kh, Vh, mask)
print(out_h.transpose(0, 1).reshape(T, 8).shape)